# Guia de Estudos: Estruturas de Dados para SEP (Backend & RPA)

Este guia foca na implementação de conceitos clássicos de **Stevenson** (Análise de Sistemas de Potência) e **Sadiku** (Elementos de Eletromagnetismo/Circuitos) utilizando programação moderna para Backend e Automação.

**Objetivo:** Modelar componentes elétricos, resolver matrizes de rede e equações diferenciais (Swing Equation) em Python, Rust e Julia.

**Objetivo:** Criar um sistema que lê um arquivo CSV (dados de barras e linhas), monta a Ybus e simula uma falta.

1. **Camada de Dados (Data Classes/Structs):**
    * Crie structs para `Bus` (ID, V, Ang) e `Branch` (From, To, R, X).
2. **Camada de Lógica (Hash Maps & Math):**
    * Leia o CSV. Use um Hash Map onde a *Key* é o ID da barra e o *Value* é o índice dela na matriz matricial (0, 1, 2...).
    * Monte a matriz `Ybus` (números complexos).
3. **Camada RPA/API (Python ou Rust):**
    * **Python (Flask):** Crie um endpoint `/simulate` que recebe um JSON com uma contingência ("Remover linha 2-3") e retorna as novas tensões.
    * **Rust:** Escreva o "solver" da rede em Rust para performance e chame-o no Python via biblioteca `PyO3` ou compile como um binário CLI.

**Leitura Recomendada:**

* *Stevenson:* Capítulos sobre "Modelagem de Redes" e "Fluxo de Potência".
* *Sadiku:* Capítulos sobre "Equações de Maxwell" (para entender de onde vem o L e C da linha).


---

## 1. Modelagem de Dados: Data Classes e Structs

*Conceito:* Como representar uma **Linha de Transmissão** ou uma **Barra (Bus)** no código. Em vez de usar listas soltas `[100, 0.05, 0.1]`, usamos estruturas nomeadas e tipadas.

### Cenário SEP: Modelo Pi de uma Linha Curta/Média ($Z = R + jX$)

#### 🐍 Python (Data Classes)

Ideal para prototipagem rápida e scriptagem no RPA.


In [10]:
from dataclasses import dataclass


@dataclass
class TransmissionLine:
    from_bus: int
    to_bus: int
    r_pu: float  # Resistência
    x_pu: float  # Reatância

    @property
    def impedance(self):
        return complex(self.r_pu, self.x_pu)


# Uso
line1 = TransmissionLine(1, 2, 0.01, 0.05)
print(f"Z: {line1.impedance}")

Z: (0.01+0.05j)


#### 🦀 Rust (Structs)

Ideal para Backend de alta performance (API) e segurança de memória.

```rust
struct TransmissionLine {
    from_bus: u32,
    to_bus: u32,
    r_pu: f64,
    x_pu: f64,
}

impl TransmissionLine {
    // Método para calcular impedância magnitude
    fn impedance_mag(&self) -> f64 {
        (self.r_pu.powi(2) + self.x_pu.powi(2)).sqrt()
    }
}
```

#### 🟣 Julia (Structs)

A melhor para cálculo numérico pesado.

```julia
struct TransmissionLine
    from_bus::Int
    to_bus::Int
    r_pu::Float64
    x_pu::Float64
end

# Funções operam sobre a struct (Multiple Dispatch)
impedance(l::TransmissionLine) = l.r_pu + im*l.x_pu
```


## 2. Hash Maps (Dicionários) & Hash Keys

*Conceito:* Acesso rápido ($O(1)$). Em SEP, os identificadores das barras (Bus ID) nem sempre são sequenciais (ex: Barra 101, Barra 504). Não podemos usar arrays simples.

### Cenário SEP: Montagem da Matriz Admitância (Ybus) Esparsa

* **Chave (Key):** ID da Barra (Int ou String).
* **Valor (Value):** Objeto `Barra` contendo tensão, ângulo, carga, geração.


#### 🦀 Rust (`HashMap`)

```rust
use std::collections::HashMap;

let mut network_buses = HashMap::new();
network_buses.insert(101, BusStruct { type: "PV", v: 1.05 });

// match é usado para lidar com o caso da barra não existir (Option)
match network_buses.get(&101) {
    Some(bus) => println!("Tensão: {}", bus.v),
    None => println!("Barra não encontrada"),
}
```

In [11]:
network_buses = {
    101: {"type": "PV", "v": 1.05, "p_gen": 50},
    205: {"type": "PQ", "load": 100},
}
# Acesso instantâneo
bus_data = network_buses.get(101)
print(bus_data)

{'type': 'PV', 'v': 1.05, 'p_gen': 50}


## 3. Pilhas (Stacks)

*Conceito:* LIFO (Last In, First Out). Útil para "navegar" na rede elétrica (Busca em Profundidade - DFS) ou gerenciar estados em um RPA (Desfazer ação).

### Cenário RPA: Histórico de Alterações de Topologia

Imagine um script que desliga disjuntores para simular contingências (N-1). Uma pilha guarda o estado anterior para restaurar a rede.

#### 🦀 Rust (`Vec`)

Rust usa Vetores como pilhas.

```rust
let mut actions: Vec<&str> = Vec::new();
actions.push("OPEN_BREAKER_A");
if let Some(last) = actions.pop() {
    println!("Desfazendo: {}", last);
}
```


In [12]:
action_stack = []

# Simula desligamento
action_stack.append("OPEN_LINE_1-2")
print("Linha 1-2 Aberta")

# Reverte (Pop)
last_action = action_stack.pop()
print(f"Revertendo: {last_action}")

Linha 1-2 Aberta
Revertendo: OPEN_LINE_1-2


## 4. Matrizes e EDOs (SymPy & Numérico)

*Conceito:* Resolver Fluxo de Potência (Sistemas Lineares/Não-Lineares) e Estabilidade Transitória (Equações Diferenciais).

### Cenário SEP: A Equação de Swing (Oscilação)

$$ M \frac{d^2\delta}{dt^2} = P_m - P_{max} \sin(\delta) $$

#### 🐍 Python (SymPy) - Simbólico

Ótimo para derivar as equações que serão usadas no código final.

In [14]:
from sympy import symbols, Function, dsolve, sin, Eq

t = symbols('t')
delta = Function('delta')(t)
M, Pm, Pmax = symbols('M Pm Pmax')

# Definindo a EDO
edo = Eq(M * delta.diff(t, t), Pm - Pmax * sin(delta))
# O SymPy tenta encontrar solução analítica, mas para SEP usamos numérico (scipy)
edo

Eq(M*Derivative(delta(t), (t, 2)), Pm - Pmax*sin(delta(t)))


#### 🟣 Julia (DifferentialEquations.jl) - O Rei da Solução Numérica

Para resolver a estabilidade de um sistema de 1000 barras, Julia é imbatível.

```julia
using DifferentialEquations

function swing_equation!(du, u, p, t)
    delta, omega = u
    M, Pm, Pmax = p
    du[1] = omega
    du[2] = (Pm - Pmax * sin(delta)) / M
end

u0 = [0.5, 0.0] # Delta inicial, Omega inicial
tspan = (0.0, 10.0)
p = (1.0, 1.0, 2.0) # Parâmetros M, Pm, Pmax
prob = ODEProblem(swing_equation!, u0, tspan, p)
sol = solve(prob)
```
